In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sys
import os
from netCDF4 import Dataset, date2num
import glob
import scipy.io
import tqdm
import xarray as xr
import os
import warnings
import matplotlib.pyplot as plt
import  scipy as sc
from pydsstools.heclib.dss import HecDss
from scipy.stats import pearsonr
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.feature import ShapelyFeature
from cartopy.io.shapereader import Reader
from sklearn import metrics
import seaborn as sns
import pickle
import spotpy
warnings.filterwarnings('ignore')

In [3]:
from osgeo import gdal, ogr
import os
import matplotlib.pyplot as plt
import tqdm
import shapely
from shapely.geometry import MultiPolygon, Point
from scipy.interpolate import griddata
from datetime import datetime
import glob
import os, shutil
from scipy import stats
import urllib.request
import time
from shapely.geometry.multipolygon import MultiPolygon
from shapely.geometry.polygon import Polygon
from shapely import wkt
from osgeo.gdalnumeric import *
from osgeo.gdalconst import *
from pandas.io.json import json_normalize
gdal.SetConfigOption("GDALWARP_IGNORE_BAD_CUTLINE", "YES")
import yaml

In [4]:
from funciones_cambio_climatico import *

In [5]:
modelos=[['CNRM-CERFACS-CNRM-CM5','SMHI-RCA4','r1i1p1'],
         ['CNRM-CERFACS-CNRM-CM5','KNMI-RACMO22E','r1i1p1'],
         ['ICHEC-EC-EARTH','SMHI-RCA4','r12i1p1'],
         ['ICHEC-EC-EARTH','KNMI-RACMO22E','r1i1p1'],
         ['ICHEC-EC-EARTH','KNMI-RACMO22E','r12i1p1'],
         ['ICHEC-EC-EARTH','DMI-HIRHAM5','r3i1p1'],
         ['ICHEC-EC-EARTH','CLMcom-CCLM4-8-17','r12i1p1'],
         ['IPSL-IPSL-CM5A-MR','SMHI-RCA4','r1i1p1'],
         ['IPSL-IPSL-CM5A-MR','IPSL-WRF381P','r1i1p1'],
         ['MOHC-HadGEM2-ES','SMHI-RCA4','r1i1p1'],
         ['MOHC-HadGEM2-ES','DMI-HIRHAM5','r1i1p1'],
         ['MOHC-HadGEM2-ES','CLMcom-CCLM4-8-17','r1i1p1'],
         ['MPI-M-MPI-ESM-LR','SMHI-RCA4','r1i1p1'],
         ['MPI-M-MPI-ESM-LR','CLMcom-CCLM4-8-17','r1i1p1'],
         ['NCC-NorESM1-M','DMI-HIRHAM5','r1i1p1']]

In [6]:
names_model=list()
for i in modelos:
    names_model.append(i[0]+'_'+i[1]+'_'+i[2])

In [7]:
name_columns = list()
for i in np.arange(1,13):
    name_columns.append(i)
    name_columns.append('Factor_'+str(i))

# Configuración del modelo

In [8]:
os.chdir('E:/Github/HEC-HMS/')

In [9]:
from funciones_HMS import *

In [10]:
Path_model ='E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/'
file_hms = 'Oued_Mellah_Python.hms'
file_gage = 'Oued_Mellah_Python.gage'
file_basin = 'Oued_Mellah_SED.basin'
file_run= 'Oued_Mellah_Python.run'

In [11]:
names_stations = read_gages(Path_model, file_gage)

In [12]:
names_control = read_control(Path_model, file_hms)

In [13]:
names_control

['ControlCP', 'ControlMP', 'ControlLP']

In [14]:
# NOMBRES DE FICHEROS -------------------------------------------------------------------------------------------------------------------------------------------------
Path_model ='E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/'
path_rain = 'E:/TUNEZ/07_Cambio_Climatico/Archivo_temporal.csv' # El fichero debe contener una columna con fecha (index) y los datos de lluvias de estaciónes en cada columna 
name_model = 'Oued_Mellah_Python' # Nombre del fichero .hms sin la extensión
name_basin = 'Oued_Mellah_SED_CP' # Igual al arrojado en namesbasin
name_control = 'Campiazo' # Igual al arrojado en namecontrol
file_dss = 'Oued_Mellah_Python.dss' # Nombre de fichero donde se almacenan las lluvias

# DEFINICIÓN DEL TIEMPO E INTERVALOS PARA LLENAR LOS DATOS DE LLUVIAS EN EL PROGRAMA -------------------------------------------------------------------------------------------------
#Nota: Si el control ya esta generado en el programa, no es necesario llenar estos datos
Time_interval = '1DAY' #(1MIN, 2MIN, 3MIN,4MIN,5MIN,6MIN,10MIN,...,1HOUR,....1DAY) dependiendo de la resolución de los datos se puede variar
Time_interval_c ='60' # Se debe ingresar en minutos, en este caso se usa un dia = 60*24 = 1440

#Name_run = 'Run' # en caso de generar una nueva corrida se asigna el nombre que prefiera el usuario

In [15]:
Centroides = pd.read_csv('E:/TUNEZ/GIS/Centroides.csv',index_col=0)

In [16]:
names_stations =  Centroides.name.values

In [17]:
dict_CC_PREC_QM_CORDEX   =  pickle.load(open('E:/TUNEZ/07_Cambio_Climatico/CLIMA/Diccionarios/'+'dict_CC_PREC_QM_CORDEX.pickle','rb'))
dict_CC_TASMAX_QM_CORDEX =  pickle.load(open('E:/TUNEZ/07_Cambio_Climatico/CLIMA/Diccionarios/'+'dict_CC_TASMAX_QM_CORDEX.pickle','rb'))
dict_CC_TASMIN_QM_CORDEX =  pickle.load(open('E:/TUNEZ/07_Cambio_Climatico/CLIMA/Diccionarios/'+'dict_CC_TASMIN_QM_CORDEX.pickle','rb'))

In [18]:
sim = 0
for nmodel, m in enumerate(names_model):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if per[2] == 'CP' and rcp=='rcp45':
                name_basin = 'Oued_Mellah_SED_CP'
                
            elif per[2] == 'CP' and rcp=='rcp85':
                name_basin = 'Oued_Mellah_SED_CP'
                
            elif per[2] == 'MP' and rcp=='rcp45':
                name_basin = 'Oued_Mellah_SED_MP'
                
            elif per[2] == 'MP' and rcp=='rcp85':
                name_basin = 'Oued_Mellah_SED_MP'
                
            elif per[2] == 'LP' and rcp=='rcp45':
                name_basin = 'Oued_Mellah_SED_LP'
                
            elif per[2] == 'LP' and rcp=='rcp85':
                name_basin = 'Oued_Mellah_SED_LP'

            yearini = per[0]
            yearfin = per[1]
            
            names_stations = list()
            for name in Centroides.name:
                names_stations.append(name+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1))
            name_met = rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
            
            Serie_Prec = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            Serie_Temp_max = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            Serie_Temp_min = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            Serie_ET = pd.DataFrame(index=pd.date_range(start=str(yearini)+'-01-01 12:00:00',end=str(yearfin)+'-12-31 12:00:00',freq='D'),columns=Centroides.name)
            
            ETP_H = pd.read_csv('E:/TUNEZ/01_Datos_Climaticos/Reconstruccion/'+ 'ETP_Centroides_2.csv',index_col=0, decimal=',',delimiter=';').iloc[:,0::2]
            ETP_T = pd.read_csv('E:/TUNEZ/01_Datos_Climaticos/Reconstruccion/'+ 'ETP_Thorw_Centroides_2.csv',index_col=0,delimiter=';', decimal=',').iloc[:,0::2]
            
            Datos_mensuales = pd.DataFrame(index=np.arange(1,13),columns=Centroides.name)
            
            for c, cc in enumerate(Centroides.name):
                Serie_Prec.loc[:,cc]     = dict_CC_PREC_QM_CORDEX[str(yearini)+'_'+str(yearfin)][rcp][m][cc].values
                Serie_Temp_max.loc[:,cc] = dict_CC_TASMAX_QM_CORDEX[str(yearini)+'_'+str(yearfin)][rcp][m][cc].values
                Serie_Temp_min.loc[:,cc] = dict_CC_TASMIN_QM_CORDEX[str(yearini)+'_'+str(yearfin)][rcp][m][cc].values
                
                tmin = Serie_Temp_min.loc[:,cc]
                tmax = Serie_Temp_max.loc[:,cc]
                tmed = (tmin+tmax)/2
                lat  = Centroides[Centroides.name==cc].loc[:,'POINT_Y'].values
                date = i
                Datos_mensuales.loc[:,cc] = thornthwaite_(tmin.astype(float),tmax.astype(float),lat)
                
            Factor = Datos_mensuales.values/ETP_T.T.values
    
            Serie_Prec.columns = names_stations
            Serie_Prec.to_csv('E:/TUNEZ/07_Cambio_Climatico/Archivo_temporal.csv')
                
            ETP_Oued_Mellah_month=Serie_ET.loc[:,Centroides.name].resample('M').sum()
            ETP_Oued_Mellah_month[ETP_Oued_Mellah_month==0]=np.nan
            
            grouped_med= ETP_Oued_Mellah_month.groupby(lambda x: x.month)
            Datos_mensuales=grouped_med.mean()
            
            Datos_mensuales_table = pd.DataFrame(columns=name_columns,index=Centroides.name)
            Datos_mensuales_table.iloc[:,:] = 2.4
            Datos_mensuales_table.loc[Centroides.name,np.arange(1,13)] = ETP_H.values*Factor.T
            Datos_mensuales_table.columns = Datos_mensuales_table.columns.astype(str)
            
            
                
            Start_Time = '1 January '+str(yearini)+', 00:00' # Fecha de inicio del control
            End_Time   = '31 December '+str(yearfin)+', 00:00'
            
            if sim==0:
                generate_gage(name_model, names_stations, Time_interval, Path_model, Start_Time, End_Time, file_dss,exists_gage=False)
            else:
                generate_gage(name_model, names_stations, Time_interval, Path_model, Start_Time, End_Time, file_dss, exists_gage=True)
            
            fill_gage(names_stations, path_rain, Time_interval, Path_model, file_dss, Start_Time, End_Time)
            generate_met(name_met, Centroides.name,names_stations,Path_model, name_basin,Evapotranspiration=True,ET_Table=Datos_mensuales_table)
            
            
            if sim==0:
                generate_run(Path_model, name_model, 'Run_'+name_met,name_met, name_basin,'Control'+per[2],exists_run=False)
            else:
                generate_run(Path_model, name_model, 'Run_'+name_met,name_met, name_basin,'Control'+per[2],exists_run=True)  
            sim = sim+1

################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:03<00:00,  9.45it/s]


################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:03<00:00,  8.98it/s]


################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


...

################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:11<00:00,  2.64it/s]


################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .gage fue modificado satisfactoriamente ###################


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:11<00:00,  2.60it/s]

################### Se ha modificado el fichero .dss satisfactoriamente y se han llenado los datos de lluvia ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################


In [19]:
names_control = read_control(Path_model, file_hms)

In [20]:
names_met =list()
import os
for file in os.listdir(Path_model):
    if file.endswith(".met"):
        names_met.append(file[:-4])

In [21]:
generate_hms(name_model, Path_model, names_met, file_dss, ['Oued_Mellah_SED_CP',
                                                           'Oued_Mellah_SED_MP',
                                                           'Oued_Mellah_SED_LP',], names_control)

################### El fichero .hms fue modificado satisfactoriamente ###############################


In [23]:
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]: #[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']
            print('Simulando '+rcp+' '+per[2]+' '+'Modelo '+ str(nmodel+1))
            Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
            Generate_py(Path_model, name_model, [Name_run.replace("_", " ")])
            
            # Con este modulo se corre el programa HEC HMS, para saber si la modelacion tuvo exito debe aparecer un 0 al final, en caso contrario aparece un -1
            os.chdir('C:/Program Files/HEC/HEC-HMS/4.9/')
            os.system('HEC-HMS.cmd -script '+Path_model+'scripts/compute_current.py')

  0%|                                                                                           | 0/15 [00:00<?, ?it/s]

Simulando rcp45 CP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 1
################### El fichero .py fue creado satisfactoriamente ###############################


  7%|█████▏                                                                        | 1/15 [59:11<13:48:40, 3551.43s/it]

Simulando rcp45 CP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 2
################### El fichero .py fue creado satisfactoriamente ###############################


 13%|██████████▏                                                                 | 2/15 [1:58:21<12:49:15, 3550.41s/it]

...

 87%|█████████████████████████████████████████████████████████████████          | 13/15 [12:28:11<1:52:51, 3385.76s/it]

Simulando rcp45 CP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 14
################### El fichero .py fue creado satisfactoriamente ###############################


 93%|███████████████████████████████████████████████████████████████████████▊     | 14/15 [13:26:15<56:55, 3415.44s/it]

Simulando rcp45 CP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 MP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp45 LP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 CP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 MP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################
Simulando rcp85 LP Modelo 15
################### El fichero .py fue creado satisfactoriamente ###############################


100%|█████████████████████████████████████████████████████████████████████████████| 15/15 [14:10:36<00:00, 3402.44s/it]


In [27]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]: #,[2041,2070,'MP'],[2071,2100,'LP']
            if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/Caudal_Basin_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv'):
                continue
            else:
                print(m + ' '+ rcp+' '+per[2])
                Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
                Caudal = pd.DataFrame(index=pd.date_range(start = str(per[0])+'-01-01 00:00:00', end=str(per[1])+'-12-31 23:00:00',freq = '1h'),columns=Centroides.name)
                Caudal_SED = pd.DataFrame(index=pd.date_range(start = str(per[0])+'-01-01 00:00:00', end=str(per[1])+'-12-31 23:00:00',freq = '1h'),columns=Centroides.name)
                fid = HecDss.Open(Path_model+Name_run+'.dss')

                for i in tqdm.tqdm(Centroides.name):
                    pn = '//'+i.upper()+'/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q = pd.DataFrame(index= times, columns = ['flow'])
                    Q.loc[:,'flow'] = values.iloc[:,0].values

                    Caudal.loc[:,i] = values.iloc[:,0].values

                    pn_s = '//'+i.upper()+'/SEDIMENT LOAD/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_SED = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_SED.values)
                    times = np.array(ts_SED.pytimes)
                    SED = pd.DataFrame(index= times, columns = ['Ton'])
                    SED.loc[:,'Ton'] = values.iloc[:,0].values.reshape(-1,1)
                    SED[SED>10000] = np.nan
                    SED[SED==-float('Inf')] = np.nan
                    SED[SED==float('Inf')] = np.nan
                    SED[SED<0] = np.nan
                    Caudal_SED.loc[SED.index,i] = SED.iloc[:,0].values

             
                Caudal.resample('D').mean().to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/Caudal_Basin_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                Caudal_SED.resample('D').sum().to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SED_Basin_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')

                pn = '//REACH3/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                values = pd.DataFrame(ts.values)
                times = np.array(ts.pytimes)
                Q = pd.DataFrame(index= times, columns = ['flow'])
                Q.loc[:,'flow'] = values.iloc[:,0].values


                pn_s = '//REACH3/SEDIMENT-OUT/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                ts_SED = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                values = pd.DataFrame(ts_SED.values)
                times = np.array(ts_SED.pytimes)
                SED = pd.DataFrame(index= times, columns = ['Ton'])
                SED.loc[:,'Ton'] = values.iloc[:,0].values.reshape(-1,1)
                SED[SED>10000] = np.nan
                SED[SED==-float('Inf')] = np.nan
                SED[SED==float('Inf')] = np.nan
                SED[SED<0] = np.nan

                Q.resample('D').mean().to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/Caudal_REACH3_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                SED.resample('D').sum().to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SED_REACH3_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')


                fid.close()

  0%|                                                                                           | 0/15 [00:00<?, ?it/s]

CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 CP



100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [11:33<00:00, 22.36s/it]


CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 MP



100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [11:32<00:00, 22.35s/it]


...


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [08:50<00:00, 17.11s/it]


NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp85 MP



100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [08:49<00:00, 17.08s/it]


NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp85 LP



100%|█████████████████████████████████████████████████████████████████████████████| 15/15 [17:23:34<00:00, 4174.30s/it]


In [28]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SED_JUNT7_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv'):
                continue
            else:
                
                print(m + ' '+ rcp+' '+per[2])
                Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
                fid = HecDss.Open(Path_model+Name_run+'.dss')


                pn = '//JUNT7/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                values = pd.DataFrame(ts.values)
                times = np.array(ts.pytimes)
                Q = pd.DataFrame(index= times, columns = ['flow'])
                Q.loc[:,'flow'] = values.iloc[:,0].values


                pn_s = '//JUNT7/SEDIMENT-OUT/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                ts_SED = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                values = pd.DataFrame(ts_SED.values)
                times = np.array(ts_SED.pytimes)
                SED = pd.DataFrame(index= times, columns = ['Ton'])
                SED.loc[:,'Ton'] = values.iloc[:,0].values.reshape(-1,1)
                SED[SED>10000] = np.nan
                SED[SED==-float('Inf')] = np.nan
                SED[SED==float('Inf')] = np.nan
                SED[SED<0] = np.nan

                Q.resample('D').mean().to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/Caudal_JUNT7_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                SED.resample('D').sum().to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SED_JUNT7_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                
                


                fid.close()

  0%|                                                                                           | 0/15 [00:00<?, ?it/s]

CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 CP
CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 MP
CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp45 LP
CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp85 CP
CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp85 MP
CNRM-CERFACS-CNRM-CM5_SMHI-RCA4_r1i1p1 rcp85 LP


  7%|█████▍                                                                            | 1/15 [01:52<26:12, 112.31s/it]

CNRM-CERFACS-CNRM-CM5_KNMI-RACMO22E_r1i1p1 rcp45 CP
CNRM-CERFACS-CNRM-CM5_KNMI-RACMO22E_r1i1p1 rcp45 MP
CNRM-CERFACS-CNRM-CM5_KNMI-RACMO22E_r1i1p1 rcp45 LP
CNRM-CERFACS-CNRM-CM5_KNMI-RACMO22E_r1i1p1 rcp85 CP
CNRM-CERFACS-CNRM-CM5_KNMI-RACMO22E_r1i1p1 rcp85 MP
CNRM-CERFACS-CNRM-CM5_KNMI-RACMO22E_r1i1p1 rcp85 LP


 13%|██████████▉                                                                       | 2/15 [03:43<24:09, 111.49s/it]

...

 87%|██████████████████████████████████████████████████████████████████████▏          | 13/15 [24:03<03:41, 110.63s/it]

MPI-M-MPI-ESM-LR_CLMcom-CCLM4-8-17_r1i1p1 rcp45 CP
MPI-M-MPI-ESM-LR_CLMcom-CCLM4-8-17_r1i1p1 rcp45 MP
MPI-M-MPI-ESM-LR_CLMcom-CCLM4-8-17_r1i1p1 rcp45 LP
MPI-M-MPI-ESM-LR_CLMcom-CCLM4-8-17_r1i1p1 rcp85 CP
MPI-M-MPI-ESM-LR_CLMcom-CCLM4-8-17_r1i1p1 rcp85 MP
MPI-M-MPI-ESM-LR_CLMcom-CCLM4-8-17_r1i1p1 rcp85 LP


 93%|███████████████████████████████████████████████████████████████████████████▌     | 14/15 [25:54<01:50, 110.87s/it]

NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp45 CP
NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp45 MP
NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp45 LP
NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp85 CP
NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp85 MP
NCC-NorESM1-M_DMI-HIRHAM5_r1i1p1 rcp85 LP


100%|█████████████████████████████████████████████████████████████████████████████████| 15/15 [27:40<00:00, 110.70s/it]


# Régimen Extremal

In [17]:
# DEFINICIÓN DEL TIEMPO E INTERVALOS PARA EL MODULO DEL CONTROL DEL PROGRAMA ---------------------------------------------------------------------------------------------------------
Path_model = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC_Extremal/'
Start_Time_c = '1 January 2021' # un intervalo antes al inicio de la lluvia (datos diarios) o el mismo dia de inicio del evento si el mismo no supera un dia 
End_Time_c = '03 January 2021'
Time_interval_c ='5' # Se debe ingresar en minutos, en este caso se usa un dia = 60*24 = 1440
file_dss = 'Oued_Mellah_Python.dss'

In [18]:
name_model = 'Oued_Mellah_Python'

In [19]:
names_sbasin= read_subbasin(Path_model, 'Oued_Mellah_SED_CP.basin')

In [20]:
Centroides = pd.read_csv('E:/TUNEZ/GIS/Centroides.csv',index_col=0)

In [21]:
for nmodel, m in enumerate(names_model):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if per[2] == 'CP':
                name_basin = 'Oued_Mellah_SED_CP'
            elif per[2] == 'MP':
                name_basin = 'Oued_Mellah_SED_MP'
            elif per[2] == 'LP':
                name_basin = 'Oued_Mellah_SED_LP'
            name_met = rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
            IDF = pd.DataFrame(index=[5/60,15/60,1,2,3,6,12,24],columns=names_sbasin)
            for T in [2,5,10,20,50,100,500] :
                for i in names_sbasin:
                    IDF_curve    = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CLIMA/IDF/'+'Curva_IDF_'+rcp+'_'+str(per[0])+'_'+str(per[1])+'_'+m+'_''Centroide_'+i+'_2.csv',index_col=0,sep=';',decimal=',')
                    IDF.loc[:,i] = IDF_curve.loc[:,'T='+str(T)].values
                generate_met_freq_storn('Hidro_T'+str(T)+'_'+name_met, names_sbasin, Path_model,IDF.astype(float), name_basin) 

################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto satisfactoriamente ###############################
################### Los ficheros .met fueron creados en la carpeta del proyecto sat

In [22]:
generate_control(name_model, Path_model, 'Control_Extremal', Start_Time_c, End_Time_c, Time_interval_c)

################### El fichero .control fue creado satisfactoriamente ###############################


In [23]:
names_met =list()
import os
for file in os.listdir(Path_model):
    if file.endswith(".met"):
        names_met.append(file[:-4])

In [24]:
generate_hms(name_model, Path_model, names_met, file_dss, ['Oued_Mellah_SED_CP','Oued_Mellah_SED_MP','Oued_Mellah_SED_LP'], ['Control_Extremal'])

################### El fichero .hms fue modificado satisfactoriamente ###############################


In [26]:
sim = 0
for nmodel, m in enumerate(names_model):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if per[2] == 'CP':
                name_basin = 'Oued_Mellah_SED_CP'
            elif per[2] == 'MP':
                name_basin = 'Oued_Mellah_SED_MP'
            elif per[2] == 'LP':
                name_basin = 'Oued_Mellah_SED_LP'
            name_met = rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
            for i,T in enumerate([2,5,10,20,50,100,500]) :
                Name_run = 'T'+str(T)+'_'+name_met
                if sim ==0:
                    generate_run(Path_model, name_model, Name_run,'Hidro_T'+str(T)+'_'+name_met, name_basin,'Control_Extremal',exists_run=False)
                else:
                    generate_run(Path_model, name_model, Name_run,'Hidro_T'+str(T)+'_'+name_met, name_basin,'Control_Extremal',exists_run=True)


                Generate_py(Path_model, name_model, [Name_run.replace("_", " ")])

                # Con este modulo se corre el programa HEC HMS, para saber si la modelacion tuvo exito debe aparecer un 0 al final, en caso contrario aparece un -1
                os.chdir('C:/Program Files/HEC/HEC-HMS/4.9/')
                os.system('HEC-HMS.cmd -script '+Path_model+'scripts/compute_current.py')
                sim = sim +1

################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .py fue creado satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .py fue creado satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .py fue creado satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .py fue creado satisfactoriamente ###############################
################### El fichero .run fue creado satisfactoriamente ###############################
################### El fichero .py fue creado satisfactoriamente ###############################
################### El fi

# Export Results

## Período Histórico Caudal Líquido

### Caudal Extremal Junt 7

In [27]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CAL/'
Caudal      = pd.DataFrame(index=Centroides.name,columns=[2,5,10,20,50,100,500])
Caudal_JUNT = pd.DataFrame(index=['JUNT7'],columns=[2,5,10,20,50,100,500])
for i,T in enumerate([2,5,10,20,50,100,500]):
    if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
        os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')
    Name_run =  'T'+str(T)

    fid = HecDss.Open(Path_model+Name_run+'.dss')

    for b in tqdm.tqdm(Centroides.name):
        pn = '//'+b.upper()+'/FLOW/01JAN2021/5MIN/RUN:'+Name_run
        ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
        values = pd.DataFrame(ts.values)
        times = np.array(ts.pytimes)
        Q = pd.DataFrame(index= times, columns = ['flow'])
        Q.loc[:,'flow'] = values.iloc[:,0].values

        Caudal.loc[b,T] = Q.max().values[0]
        
    pn = '//JUNT7/FLOW/01JAN2021/5MIN/RUN:'+Name_run
    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
    values = pd.DataFrame(ts.values)
    times = np.array(ts.pytimes)
    Q = pd.DataFrame(index= times, columns = ['flow'])
    Q.loc[:,'flow'] = values.iloc[:,0].values
    
    Caudal_JUNT.loc[:,T] = Q.max().values[0]
    
    fid.close()  
    
Caudal.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_Hist_2.csv')
        
Caudal_JUNT.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_JUNT7_Hist_2.csv')

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:00<00:00, 32.45it/s]


In [28]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC_Extremal/'
Caudal      = pd.DataFrame(index=Centroides.name,columns=[2,5,10,20,50,100,500])
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            Caudal_JUNT7 = pd.DataFrame(index=pd.date_range(start = '2021-01-01', end='2021-01-03',freq = '5min'),columns=[2,5,10,20,50,100,500])
            for i,T in enumerate([2,5,10,20,50,100,500]):
                if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
                    os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')

                Name_run =  'T'+str(T)+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)

                fid = HecDss.Open(Path_model+Name_run+'.dss')
                for b in tqdm.tqdm(Centroides.name):
                    pn = '//'+b.upper()+'/FLOW/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q = pd.DataFrame(index= times, columns = ['flow'])
                    Q.loc[:,'flow'] = values.iloc[:,0].values

                    Caudal.loc[b,T] = Q.max().values[0]

                pn = '//JUNT7/FLOW/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                values = pd.DataFrame(ts.values)
                values[values<0] = np.nan
                times = np.array(ts.pytimes)
                
                Caudal_JUNT7.loc[:,T] = values.iloc[:,0].values
                fid.close()      
            Caudal_JUNT7.to_csv('E:/TUNEZ/09_Envio_Resultados/Caudal/Extremal/CC/Hidrogramas_Reg_Extremal_Ouchtata_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
            Caudal_JUNT7.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_JUNT7_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
            Caudal.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Basin/Caudal_Extremal_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:00<00:00, 32.98it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:00<00:00, 34.52it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:01<00:00, 29.35it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:00<00:00, 31.45it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:01<00:00, 28.13it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:01<00:00, 29.43it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:00<00:00, 33.45it/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:01<00:00, 28.27it/s]

100%|███████████████████████████

### Caudal Reach 3

In [29]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CAL/'
Caudal_JUNT = pd.DataFrame(index=['REACH3'],columns=[2,5,10,20,50,100,500])
for i,T in enumerate([2,5,10,20,50,100,500]):
    if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
        os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')
    Name_run =  'T'+str(T)

    fid = HecDss.Open(Path_model+Name_run+'.dss')

        
    pn = '//REACH3/FLOW/01JAN2021/5MIN/RUN:'+Name_run
    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
    values = pd.DataFrame(ts.values)
    times = np.array(ts.pytimes)
    Q = pd.DataFrame(index= times, columns = ['flow'])
    Q.loc[:,'flow'] = values.iloc[:,0].values
    
    Caudal_JUNT.loc[:,T] = Q.max().values[0]
    
    fid.close()  
        
Caudal_JUNT.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_REACH3_Hist_2.csv')

In [17]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC_Extremal/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            Caudal_REACH3 = pd.DataFrame(index=pd.date_range(start = '2021-01-01', end='2021-01-03',freq = '5min'),columns=[2,5,10,20,50,100,500])
            for i,T in enumerate([2,5,10,20,50,100,500]):
                if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
                    os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')

                Name_run =  'T'+str(T)+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)

                fid = HecDss.Open(Path_model+Name_run+'.dss')

                pn = '//REACH3/FLOW/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                values = pd.DataFrame(ts.values)
                values[values<0] = np.nan
                times = np.array(ts.pytimes)
                
                Caudal_REACH3.loc[:,T] = values.iloc[:,0].values
                fid.close()      
            Caudal_REACH3.to_csv('E:/TUNEZ/09_Envio_Resultados/Caudal/Extremal/CC/Hidrogramas_Reg_Extremal_Punto_Extraccion_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
            Caudal_REACH3.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_REACH3_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:38<00:00,  2.57s/it]


## Período Histórico Caudal Sólido

### Caudal Extremal Junt 7

In [18]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CAL/'
Caudal      = pd.DataFrame(index=Centroides.name,columns=[2,5,10,20,50,100,500])
Caudal_JUNT = pd.DataFrame(index=['JUNT7'],columns=[2,5,10,20,50,100,500])
for i,T in enumerate([2,5,10,20,50,100,500]):
    if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
        os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')
    Name_run =  'T'+str(T)

    fid = HecDss.Open(Path_model+Name_run+'.dss')

    for b in tqdm.tqdm(Centroides.name):
        pn = '//'+b.upper()+'/SEDIMENT LOAD/01JAN2021/5MIN/RUN:'+Name_run
        ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
        values = pd.DataFrame(ts.values)
        values[values<0] = np.nan
        times = np.array(ts.pytimes)
        Q = pd.DataFrame(index= times, columns = ['flow'])
        Q.loc[:,'flow'] = values.iloc[:,0].values

        Caudal.loc[b,T] = np.nansum(Q.values)
        
    pn = '//JUNT7/SEDIMENT-OUT/01JAN2021/5MIN/RUN:'+Name_run
    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
    values = pd.DataFrame(ts.values)
    values[values<0] = np.nan
    times = np.array(ts.pytimes)
    Q = pd.DataFrame(index= times, columns = ['flow'])
    Q.loc[:,'flow'] = values.values
    
    Caudal_JUNT.loc[:,T] = np.nansum(Q.values)
    
    fid.close()  
    
Caudal.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_SED_Hist_2.csv')
        
Caudal_JUNT.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_SED_JUNT7_Hist_2.csv')

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:00<00:00, 34.64it/s]


### Caudal Reach 3

In [19]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CAL/'
Caudal_JUNT = pd.DataFrame(index=['REACH3'],columns=[2,5,10,20,50,100,500])
for i,T in enumerate([2,5,10,20,50,100,500]):
    if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
        os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')
    Name_run =  'T'+str(T)

    fid = HecDss.Open(Path_model+Name_run+'.dss')

        
    pn = '//REACH3/SEDIMENT-OUT/01JAN2021/5MIN/RUN:'+Name_run
    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
    values = pd.DataFrame(ts.values)
    values[values<0] = np.nan
    times = np.array(ts.pytimes)
    Q = pd.DataFrame(index= times, columns = ['flow'])
    Q.loc[:,'flow'] = values.values
    
    Caudal_JUNT.loc[:,T] = np.nansum(Q.values)
    
    fid.close()  
        
Caudal_JUNT.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_SED_REACH3_Hist_2.csv')

In [20]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC_Extremal/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            Caudal = pd.DataFrame(index=Centroides.name,columns=[2,5,10,20,50,100,500])
            Caudal_JUNT = pd.DataFrame(index=['JUNT7'],columns=[2,5,10,20,50,100,500])
            for i,T in enumerate([2,5,10,20,50,100,500]):
                if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
                    os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')

                Name_run =  'T'+str(T)+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)

                fid = HecDss.Open(Path_model+Name_run+'.dss')

                for b in Centroides.name:
                    pn = '//'+b.upper()+'/SEDIMENT LOAD/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                    values = pd.DataFrame(ts.values)
                    values[values<0] = np.nan
                    times = np.array(ts.pytimes)
                    Q = pd.DataFrame(index= times, columns = ['flow'])
                    Q.loc[:,'flow'] = values.iloc[:,0].values

                    Caudal.loc[b,T] = np.nansum(Q.values)

                pn = '//JUNT7/SEDIMENT-OUT/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                values = pd.DataFrame(ts.values)
                values[values<0] = np.nan
                times = np.array(ts.pytimes)
                Q = pd.DataFrame(index= times, columns = ['flow'])
                Q.loc[:,'flow'] = values.iloc[:,0].values

                Caudal_JUNT.loc[:,T] = np.nansum(Q.values)
                fid.close()      
            Caudal.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_SED_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
            Caudal_JUNT.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_SED_JUNT7_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [10:20<00:00, 41.35s/it]


In [21]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC_Extremal/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            Caudal_JUNT = pd.DataFrame(index=['JUNT7'],columns=[2,5,10,20,50,100,500])
            for i,T in enumerate([2,5,10,20,50,100,500]):
                if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')==False:
                    os.makedirs('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/')

                Name_run =  'T'+str(T)+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)

                fid = HecDss.Open(Path_model+Name_run+'.dss')

                pn = '//REACH3/SEDIMENT-OUT/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                values = pd.DataFrame(ts.values)
                values[values<0] = np.nan
                times = np.array(ts.pytimes)
                Q = pd.DataFrame(index= times, columns = ['flow'])
                Q.loc[:,'flow'] = values.iloc[:,0].values

                Caudal_JUNT.loc[:,T] = np.nansum(Q.values)
                fid.close()      
            Caudal_JUNT.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL_EXTREMAL/Caudal_Extremal_SED_REACH3_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:27<00:00,  1.85s/it]


# Infiltration

In [23]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CAL/'
Name_run =  'Sim_1979_2005'
Caudal = pd.DataFrame(index=pd.date_range(start = '1979-01-01 00:00:00', end='2005-12-31 00:00:00',freq = '1h'),columns=Centroides.name)
fid = HecDss.Open(Path_model+Name_run+'.dss')

for i in tqdm.tqdm(Centroides.name):
    pn = '//'+i.upper()+'/INFILTRATION/01JAN1979/1HOUR/RUN:SIM_1979_2005/'
    ts = fid.read_ts(pn,window=('1979-01-01','2005-12-31'))
    values = pd.DataFrame(ts.values)
    times = np.array(ts.pytimes)
    Q = pd.DataFrame(index= times, columns = ['flow'])
    Q.loc[:,'flow'] = values.iloc[:,0].values

    Caudal.loc[Q.index,i] = values.iloc[:,0].values

Caudal.dropna().resample('D').sum().mean(axis=1).to_csv('E:/TUNEZ/07_Cambio_Climatico/INFILTRATION/Infiltration_Basin_1979_2005_2.csv')
fid.close()

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [05:43<00:00, 11.09s/it]


In [24]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/INFILTRATION/Infiltration_Basin_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')==True:
                continue
            else:
                print(m + ' '+ rcp+' '+per[2])
                Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
                Caudal = pd.DataFrame(index=pd.date_range(start = str(per[0])+'-01-01 00:00:00', end=str(per[1])+'-12-31 23:00:00',freq = '1h'),columns=Centroides.name)
                fid = HecDss.Open(Path_model+Name_run+'.dss')

                for i in tqdm.tqdm(Centroides.name):
                    pn = '//'+i.upper()+'/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q = pd.DataFrame(index= times, columns = ['flow'])
                    Q.loc[:,'flow'] = values.iloc[:,0].values

                    Caudal.loc[Q.index,i] = values.iloc[:,0].values
                fid.close()

                Caudal.dropna().resample('D').sum().mean(axis=1).to_csv('E:/TUNEZ/07_Cambio_Climatico/INFILTRATION/Infiltration_Basin_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')

100%|█████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 394.74it/s]


# Extracción de hidrogramas

In [ ]:
Juntions = pd.read_csv('E:/TUNEZ/GIS/Juntions.csv',index_col=0).Name.values

In [ ]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_Extremal_CC/'
for rcp in ['rcp45','rcp85']:
    for per in tqdm.tqdm([[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]):
        for j in Juntions:
            Caudal_Juntion = pd.DataFrame(index=pd.date_range(start = '2021-01-01', end='2021-01-03',freq = '5min'),columns=[2,5,10,20,50,100,500])
            for i,T in enumerate([2,5,10,20,50,100,500]):
                Caudal_Model = pd.DataFrame(index=pd.date_range(start = '2021-01-01', end='2021-01-03',freq = '5min'),columns=names_model)
                for nmodel, m in enumerate(names_model):
                    Name_run =  'T'+str(T)+'_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)

                    fid = HecDss.Open(Path_model+Name_run+'.dss')

                    pn =  '//'+j.upper()+'/FLOW/01JAN2021/5MIN/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=('2021-01-01','2021-01-04'))
                    values = pd.DataFrame(ts.values)
                    values[values<0] = np.nan
                    times = np.array(ts.pytimes)

                    Caudal_Model.loc[:,m] = values.iloc[:,0].values
                    fid.close()
                Caudal_Juntion.loc[:,T] = Caudal_Model.median(axis=1).values
            Caudal_Juntion.to_csv('E:/TUNEZ/09_Envio_Resultados/Caudal/Extremal/Hidrogramas/'+j+'/Hidrograma_'+rcp+'_'+str(per[0])+'_'+str(per[1])+'_'+j.upper()+'_2.csv')

## Extraer resultados Modelo Embalse

In [ ]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CC_DAM/'
for nmodel, m in enumerate(tqdm.tqdm(names_model)):
    for rcp in ['rcp45','rcp85']:
        for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
            if os.path.exists('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv'):
                continue
            else:
                try:
                    print(m + ' '+ rcp+' '+per[2])
                    Name_run =  'Run_'+rcp+'_'+per[2]+'_Model_'+str(nmodel+1)
                    fid = HecDss.Open(Path_model+Name_run+'.dss')
                    
                    pn = '//JUNT7/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q_OUCH= pd.DataFrame(index= times, columns = ['flow'])
                    Q_OUCH.loc[:,'flow'] = values.iloc[:,0].values


                    pn = '//RESERVOIR-1-SPILL-1/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts.values)
                    times = np.array(ts.pytimes)
                    Q_SPILL = pd.DataFrame(index= times, columns = ['m3/s'])
                    Q_SPILL.loc[:,'m3/s'] = values.iloc[:,0].values

                    del values

                    pn_ele = '//RESERVOIR-1/ELEVATION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_elev = fid.read_ts(pn_ele,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_elev.values)
                    times = np.array(ts_elev.pytimes)
                    ELEV = pd.DataFrame(index= times, columns = ['msnm'])
                    ELEV.loc[:,'msnm'] = values.iloc[:,0].values

                    del values

                    pn_stor = '//RESERVOIR-1/STORAGE/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_stor = fid.read_ts(pn_stor,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_stor.values)
                    times = np.array(ts_stor.pytimes)
                    STOR = pd.DataFrame(index= times, columns = ['Hm3'])
                    STOR.loc[:,'Hm3'] = values.iloc[:,0].values*10**-3

                    del values

                    pn_out_reach = '//REACH19/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_out_reach = fid.read_ts(pn_out_reach,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values_rch = pd.DataFrame(ts_out_reach.values)
                    times = np.array(ts_out_reach.pytimes)

                    pn_out_bas = '//BASIN93/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_out_bas = fid.read_ts(pn_out_bas,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values_bas = pd.DataFrame(ts_out_bas.values)
                    times = np.array(ts_out_bas.pytimes)

                    INFLOW = pd.DataFrame(index= times, columns = ['m3/s'])
                    INFLOW.loc[:,'m3/s'] = values_bas.iloc[:,0].values+values_rch.iloc[:,0].values

                    pn_dema = '//SEJNANE/FLOW-DIVERSION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
                    ts_dema = fid.read_ts(pn_dema,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
                    values = pd.DataFrame(ts_dema.values)
                    times = np.array(ts_dema.pytimes)
                    DEMA = pd.DataFrame(index= times, columns = ['Hm3'])
                    DEMA.loc[:,'Hm3'] = values.iloc[:,0].values


                    Q_SPILL.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SPILL_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    ELEV.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/ELEV_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    STOR.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/STORAGE_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    INFLOW.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    DEMA.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/DEMAND_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                    Q_OUCH.to_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv')
                except:
                    continue
                fid.close()

In [ ]:
Path_model      = 'E:/TUNEZ/Modelos_Nuevos/Oued_Mellah_CAL_DAM/'
Name_run =  'Hist_1979_2005'
fid = HecDss.Open(Path_model+'Hist_1979_2005'+'.dss')

per = [1979,2005]

pn = '//JUNT7/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.replace("_", " ")
ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts.values)
times = np.array(ts.pytimes)
Q_OUCH= pd.DataFrame(index= times, columns = ['flow'])
Q_OUCH.loc[:,'flow'] = values.iloc[:,0].values

pn = '//RESERVOIR-1-SPILL-1/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts = fid.read_ts(pn,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts.values)
times = np.array(ts.pytimes)
Q_SPILL = pd.DataFrame(index= times, columns = ['m3/s'])
Q_SPILL.loc[:,'m3/s'] = values.iloc[:,0].values

del values

pn_ele = '//RESERVOIR-1/ELEVATION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_elev = fid.read_ts(pn_ele,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts_elev.values)
times = np.array(ts_elev.pytimes)
ELEV = pd.DataFrame(index= times, columns = ['msnm'])
ELEV.loc[:,'msnm'] = values.iloc[:,0].values

del values

pn_stor = '//RESERVOIR-1/STORAGE/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_stor = fid.read_ts(pn_stor,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts_stor.values)
times = np.array(ts_stor.pytimes)
STOR = pd.DataFrame(index= times, columns = ['Hm3'])
STOR.loc[:,'Hm3'] = values.iloc[:,0].values*10**-3

del values

pn_out_reach = '//REACH19/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_out_reach = fid.read_ts(pn_out_reach,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values_rch = pd.DataFrame(ts_out_reach.values)
times = np.array(ts_out_reach.pytimes)

pn_out_bas = '//BASIN93/FLOW/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_out_bas = fid.read_ts(pn_out_bas,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values_bas = pd.DataFrame(ts_out_bas.values)
times = np.array(ts_out_bas.pytimes)

INFLOW = pd.DataFrame(index= times, columns = ['m3/s'])
INFLOW.loc[:,'m3/s'] = values_bas.iloc[:,0].values+values_rch.iloc[:,0].values

pn_dema = '//SEJNANE/FLOW-DIVERSION/01JAN'+str(per[0])+'/1HOUR/RUN:'+Name_run.upper().replace("_", " ")
ts_dema = fid.read_ts(pn_dema,window=(str(per[0])+'-01-01',str(per[1]+1)+'-12-31'))
values = pd.DataFrame(ts_dema.values)
times = np.array(ts_dema.pytimes)
DEMA = pd.DataFrame(index= times, columns = ['Hm3'])
DEMA.loc[:,'Hm3'] = values.iloc[:,0].values

Q_SPILL.columns = ['m3/s']
ELEV.columns = ['msnm']
STOR.columns = ['Hm3']
INFLOW.columns = ['m3/s']
DEMA.columns = ['m3/s']


Q_SPILL.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/SPILL_RESERVOIR_historical_2.xlsx')
ELEV.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/ELEV_RESERVOIR_historical_2.xlsx')
STOR.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/STORAGE_RESERVOIR_historical_2.xlsx')
INFLOW.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_RESERVOIR_historical_2.xlsx')
DEMA.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/DEMAND_RESERVOIR_historical_2.xlsx')
Q_OUCH.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_OUCHTATA_historical_2.xlsx')

               
fid.close()

In [ ]:
for rcp in ['rcp45','rcp85']:
    for per in [[2011,2040,'CP'],[2041,2070,'MP'],[2071,2100,'LP']]:
        n = 0
        for nmodel, m in enumerate(tqdm.tqdm(names_model)):
            if n==0:
                try:
                    Q_SPILL = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SPILL_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    ELEV = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/ELEV_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    STOR = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/STORAGE_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    INFLOW = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    DEMA = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/DEMAND_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    OUCH = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    n = n+1
                except:
                    continue
            else:
                try:
                    Q_SPILL_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/SPILL_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    ELEV_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/ELEV_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    STOR_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/STORAGE_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    INFLOW_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    DEMA_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/DEMAND_RESERVOIR_2_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)
                    OUCH_2 = pd.read_csv('E:/TUNEZ/07_Cambio_Climatico/CAUDAL/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Model_'+m+'_2.csv',index_col=0,parse_dates=True)

                    Q_SPILL = pd.concat((Q_SPILL,Q_SPILL_2),axis=1)
                    ELEV = pd.concat((ELEV,ELEV_2),axis=1)
                    STOR = pd.concat((Q_SPILL,STOR_2),axis=1)
                    INFLOW = pd.concat((INFLOW,INFLOW_2),axis=1)
                    DEMA = pd.concat((DEMA,DEMA_2),axis=1)
                    OUCH = pd.concat((OUCH,OUCH_2),axis=1)
                    n = n+1
                except:
                    continue
                
        Q_SPILL_ense = pd.DataFrame(Q_SPILL.median(axis=1))
        ELEV_ense    = pd.DataFrame(ELEV.median(axis=1))
        STOR_ense    = pd.DataFrame(STOR.median(axis=1))
        INFLOW_ense  = pd.DataFrame(INFLOW.median(axis=1))
        DEMA_ense    = pd.DataFrame(DEMA.median(axis=1))
        OUCH_ense    = pd.DataFrame(OUCH.median(axis=1))
        
        Q_SPILL_ense.columns = ['m3/s']
        ELEV_ense.columns = ['msnm']
        STOR_ense.columns = ['Hm3']
        INFLOW_ense.columns = ['m3/s']
        DEMA_ense.columns = ['m3/s']
        OUCH_ense.columns = ['m3/s']
        
        
        Q_SPILL_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/SPILL_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        ELEV_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/ELEV_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        STOR_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/STORAGE_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        INFLOW_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        DEMA_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/DEMAND_RESERVOIR_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')
        OUCH_ense.resample('D').mean().to_excel('E:/TUNEZ/07_Cambio_Climatico/GESTION/INFLOW_OUCHTATA_'+rcp+'_'+per[2]+'_Ensemble'+'_2.xlsx')